In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import umap

sys.path.insert(0, str(Path.cwd().parent))

from src.utils.h5_store import load_residue_embeddings
from src.utils.device import get_device
from src.extraction.data_loader import load_metadata_csv, read_fasta
from src.compressors.mean_pool import MeanPoolCompressor
from src.compressors.attention_pool import AttentionPoolCompressor
from src.compressors.hierarchical import HierarchicalCompressor
from src.compressors.fourier_basis import FourierBasisCompressor
from src.compressors.vq_compress import VQCompressor

DATA_DIR = Path.cwd().parent / "data"
device = get_device()
print(f"Device: {device}")

# Load data
embeddings = load_residue_embeddings(DATA_DIR / "residue_embeddings" / "esm2_8m_tiny100.h5")
metadata = load_metadata_csv(DATA_DIR / "proteins" / "metadata.csv")
embed_dim = next(iter(embeddings.values())).shape[-1]
id_to_meta = {m["id"]: m for m in metadata}
protein_ids = [pid for pid in embeddings if pid in id_to_meta]
print(f"{len(protein_ids)} proteins, embed_dim={embed_dim}")

# Protein Embedding Compression Explorer - Analysis

Comparison of baseline and novel compression strategies on ESM2-8M embeddings of ~100 SCOPe proteins.